# Question Router: TF-IDF + Logistic Regression

Train a lightweight question-only router for `chartqa`, `docvqa`, and `textvqa`, then map each predicted task to the current best backend:

- `chartqa -> chart_dora_r8_a16_B_lr2e-5`
- `docvqa -> base_zero_shot`
- `textvqa -> textvqa_lora_only`

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import sys

REPO_NAME = "multi-task-moe-vlm-assistant"
REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"


def find_project_root(clone_if_missing: bool = True) -> Path:
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content") / REPO_NAME,
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate

    colab_target = Path("/content") / REPO_NAME
    if clone_if_missing and Path("/content").exists() and not colab_target.exists():
        print(f"Project repo not found. Cloning into {colab_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(colab_target)], check=True)
        if (colab_target / "src").exists():
            return colab_target

    raise ModuleNotFoundError(
        "Could not find project root containing src/. "
        f"Clone {REPO_URL} first, or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = find_project_root(clone_if_missing=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data/processed/router/validation_questions.jsonl"
ROUTER_DIR = PROJECT_ROOT / "checkpoints/router"
ROUTER_PATH = ROUTER_DIR / "question_router.joblib"
METADATA_PATH = ROUTER_DIR / "question_router_metadata.json"
SEED = 42
TEST_SIZE = 0.2
MIN_CONFIDENCE = 0.65
FORCE_REBUILD_ROUTER_DATA = False
ROUTER_DATA_LIMITS = {"docvqa": 1536, "chartqa": 1536, "textvqa": 1536}

random.seed(SEED)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())

## Install Dependencies In Colab If Needed


In [ ]:
# Uncomment in Colab if dependencies are missing.
# %pip install -q datasets scikit-learn pandas joblib

## Prepare Question-Only Router Data


In [ ]:
# This notebook does not require image files. If gitignored processed data is
# missing, build a small question-only JSONL directly from Hugging Face.

import json
from itertools import islice


def _answer_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    return [str(value)]


def _stream_router_rows(dataset_name: str, limit: int | None, split: str = "validation"):
    from datasets import load_dataset

    if dataset_name == "docvqa":
        dataset = load_dataset("lmms-lab/DocVQA", "DocVQA", split=split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "docvqa",
                "task_type": "docvqa",
                "split": split,
                "question_id": row.get("questionId"),
                "question": row.get("question"),
                "answers": _answer_list(row.get("answers")),
            }
    elif dataset_name == "chartqa":
        hf_split = "val" if split == "validation" else split
        dataset = load_dataset("HuggingFaceM4/ChartQA", split=hf_split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "chartqa",
                "task_type": "chartqa",
                "split": split,
                "question_id": row.get("id") or row.get("id_image"),
                "question": row.get("query") or row.get("question"),
                "answers": _answer_list(row.get("label") or row.get("answer")),
            }
    elif dataset_name == "textvqa":
        dataset = load_dataset("lmms-lab/textvqa", split=split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "textvqa",
                "task_type": "textvqa",
                "split": split,
                "question_id": row.get("question_id"),
                "question": row.get("question"),
                "answers": _answer_list(row.get("answers")),
            }
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")


def build_router_question_jsonl(path: Path, limits: dict[str, int | None]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open("w", encoding="utf-8") as handle:
        for dataset_name in ["docvqa", "chartqa", "textvqa"]:
            dataset_count = 0
            for record in _stream_router_rows(dataset_name, limits.get(dataset_name)):
                question = str(record.get("question") or "").strip()
                if not question:
                    continue
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                count += 1
                dataset_count += 1
            print(f"{dataset_name}: saved {dataset_count} question records")
    return count


if FORCE_REBUILD_ROUTER_DATA or not DATA_PATH.exists():
    print("Router data missing or rebuild requested. Building question-only data:", DATA_PATH)
    print("Limits:", ROUTER_DATA_LIMITS)
    total = build_router_question_jsonl(DATA_PATH, ROUTER_DATA_LIMITS)
    print("Saved router question records:", total)
else:
    print("Using existing router data:", DATA_PATH)

## Load Questions

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_NAME = globals().get("REPO_NAME", "multi-task-moe-vlm-assistant")
REPO_URL = globals().get("REPO_URL", "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git")


def find_project_root(clone_if_missing: bool = True) -> Path:
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content") / REPO_NAME,
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate

    colab_target = Path("/content") / REPO_NAME
    if clone_if_missing and Path("/content").exists() and not colab_target.exists():
        print(f"Project repo not found. Cloning into {colab_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(colab_target)], check=True)
        if (colab_target / "src").exists():
            return colab_target

    raise ModuleNotFoundError(
        "Could not find project root containing src/. "
        f"Clone {REPO_URL} first, or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = globals().get("PROJECT_ROOT")
if PROJECT_ROOT is None or not (Path(PROJECT_ROOT) / "src").exists():
    PROJECT_ROOT = find_project_root(clone_if_missing=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = globals().get("DATA_PATH", PROJECT_ROOT / "data/processed/router/validation_questions.jsonl")

from src.data.answers import canonicalize_task_type


def load_router_records(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            question = str(row.get("question", "")).strip()
            if not question:
                continue
            task_type = canonicalize_task_type(row.get("task_type", ""))
            if task_type not in {"chartqa", "docvqa", "textvqa"}:
                continue
            records.append({"question": question, "task_type": task_type})
    return records

records = load_router_records(DATA_PATH)
print("records:", len(records))
print("task counts:", Counter(r["task_type"] for r in records))
records[:3]

## Split

In [ ]:
from sklearn.model_selection import train_test_split

questions = [r["question"] for r in records]
labels = [r["task_type"] for r in records]

train_q, test_q, train_y, test_y = train_test_split(
    questions,
    labels,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels,
)

print("train:", len(train_q), Counter(train_y))
print("test:", len(test_q), Counter(test_y))

## Train TF-IDF + Logistic Regression

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_NAME = globals().get("REPO_NAME", "multi-task-moe-vlm-assistant")
REPO_URL = globals().get("REPO_URL", "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git")


def find_project_root(clone_if_missing: bool = True) -> Path:
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content") / REPO_NAME,
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate

    colab_target = Path("/content") / REPO_NAME
    if clone_if_missing and Path("/content").exists() and not colab_target.exists():
        print(f"Project repo not found. Cloning into {colab_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(colab_target)], check=True)
        if (colab_target / "src").exists():
            return colab_target

    raise ModuleNotFoundError(
        "Could not find project root containing src/. "
        f"Clone {REPO_URL} first, or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = globals().get("PROJECT_ROOT")
if PROJECT_ROOT is None or not (Path(PROJECT_ROOT) / "src").exists():
    PROJECT_ROOT = find_project_root(clone_if_missing=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = globals().get("DATA_PATH", PROJECT_ROOT / "data/processed/router/validation_questions.jsonl")

from src.routing.task_router import (
    TfidfLogRegTaskRouter,
    build_tfidf_logreg_pipeline,
    select_task_backend,
    summarize_router_backends,
)

router = TfidfLogRegTaskRouter()
router.fit(train_q, train_y)
print(router.pipeline)

## Evaluate

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

pred_y = [router.predict(q) for q in test_q]
accuracy = accuracy_score(test_y, pred_y)
macro_f1 = f1_score(test_y, pred_y, average="macro")

print(f"accuracy: {accuracy:.4f}")
print(f"macro_f1: {macro_f1:.4f}")
print(classification_report(test_y, pred_y, digits=4))

labels_order = ["chartqa", "docvqa", "textvqa"]
cm = confusion_matrix(test_y, pred_y, labels=labels_order)
pd.DataFrame(cm, index=[f"true_{x}" for x in labels_order], columns=[f"pred_{x}" for x in labels_order])

## Backend Decisions

In [ ]:
decisions = [select_task_backend(q, router=router, min_confidence=MIN_CONFIDENCE) for q in test_q]
print("backend counts:", summarize_router_backends(decisions))

preview = []
for q, true_label, pred_label, decision in list(zip(test_q, test_y, pred_y, decisions))[:20]:
    preview.append(
        {
            "question": q,
            "true_task": true_label,
            "pred_task": pred_label,
            "backend": decision.backend_name,
            "use_adapter": decision.use_adapter,
            "adapter": decision.adapter_name,
            "confidence": decision.confidence,
        }
    )

pd.DataFrame(preview)

## Low Confidence Fallback Check

In [ ]:
for q in [
    "what is shown here?",
    "what is the invoice date?",
    "what is the highest value in 2020?",
    "what word is written on the sign?",
]:
    decision = select_task_backend(q, router=router, min_confidence=MIN_CONFIDENCE)
    print(q)
    print(" ->", decision)

## Save Router

In [ ]:
ROUTER_DIR.mkdir(parents=True, exist_ok=True)
router.save(ROUTER_PATH)

metadata = {
    "model_type": "tfidf_logistic_regression",
    "router_path": str(ROUTER_PATH),
    "data_path": str(DATA_PATH),
    "seed": SEED,
    "test_size": TEST_SIZE,
    "min_confidence": MIN_CONFIDENCE,
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "backend_policy": {
        "chartqa": "chart_dora_r8_a16_B_lr2e-5",
        "docvqa": "base_zero_shot",
        "textvqa": "textvqa_lora_only",
    },
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("saved router:", ROUTER_PATH)
print("saved metadata:", METADATA_PATH)

## Load And Predict Helper

In [ ]:
loaded_router = TfidfLogRegTaskRouter.load(ROUTER_PATH)


def predict_task(question: str, min_confidence: float = MIN_CONFIDENCE) -> dict:
    task_type, confidence = loaded_router.predict_with_confidence(question)
    decision = select_task_backend(question, router=loaded_router, min_confidence=min_confidence)
    return {
        "task_type": task_type,
        "confidence": confidence,
        "backend_name": decision.backend_name,
        "use_adapter": decision.use_adapter,
        "adapter_name": decision.adapter_name,
        "checkpoint_dir": decision.checkpoint_dir,
    }

predict_task("What is the total revenue in 2020?")